In [ ]:
#Google Colab setup for unsloth

%%capture
import os
!pip install --upgrade -qqq uv
if "COLAB_" not in "".join(os.environ.keys()):
    # If you're not in Colab, just use pip install!
    !pip install unsloth vllm
else:
    try: import numpy, PIL; _numpy = f'numpy=={numpy.__version__}'; _pil = f'pillow=={PIL.__version__}'
    except: _numpy = "numpy"; _pil = "pillow"
    try: import subprocess; is_t4 = "Tesla T4" in str(subprocess.check_output(["nvidia-smi"]))
    except: is_t4 = False
    _vllm, _triton = ('vllm==0.9.2', 'triton==3.2.0') if is_t4 else ('vllm==0.15.1', 'triton')
    !uv pip install -qqq --upgrade {_vllm} {_numpy} {_pil} torchvision bitsandbytes xformers unsloth
    !uv pip install -qqq {_triton}
    !uv pip install -qqq --no-deps --upgrade "torchao>=0.16.0"
!uv pip install transformers==4.56.2
!uv pip install --no-deps trl==0.22.2

In [ ]:
#Lightning.ai setup for unsloth

%pip uninstall -y unsloth unsloth_zoo trl transformers
%pip install --no-cache-dir "transformers==4.56.2" "trl==0.22.2"
%pip install --upgrade --force-reinstall --no-cache-dir --no-deps unsloth unsloth_zoo

In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048
lora_rank = 32
model_path = "Qwen/Qwen3-4B-Instruct-2507"  
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_path,
    max_seq_length = max_seq_length,
    load_in_4bit = True, # False for LoRA 16bit
    fast_inference = False, # Enable vllm fast inference
    max_lora_rank = lora_rank,
    gpu_memory_utilization = 0.75, # Reduce if out of memory
)


model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = lora_rank * 2, # *2 speeds up training
    use_gradient_checkpointing = "unsloth", # Reduces memory usage
    random_state = 42,
)




#### Define System Prompt

In [ ]:
# @title
system_prompt = '''# Minesweeper AI System Prompt

You are an AI agent playing Minesweeper.
Your objective is to maximize score while completing the game without revealing a mine.

## Allowed Actions

You may send exactly one move at a time using one of these actions:

- `reveal`
- `flag`

Each move must target exactly one tile coordinate:

- `x`
- `y`

Example move payload:

```json
{
  "action": "reveal",
  "x": 3,
  "y": 4
}
```

## Rules You Must Follow

- Revealing a flagged tile is invalid
- Flagging a revealed tile is invalid
- Revealing a mine ends the game immediately
- The game is won only when all safe tiles are revealed and all mines are correctly flagged
- Hidden mine locations are not exposed while the game is in progress

## Scoring Rules

- Reveal a safe tile: `+1` for each safe tile revealed
- Correctly flag a mine: `+2`
- Incorrectly flag a safe tile: `-2`
- Reveal a mine: immediate loss
- Win the full game: `+50`

## State Interpretation

You receive game state that includes:
- `board`

Symbol meanings:

- `.` means the tile is unrevealed
- `F` means the tile is flagged
- `_` means the tile is revealed and has zero adjacent mines
- `"1"` to `"8"` mean the tile is revealed and the value is the number of adjacent mines
- `B` means a bomb tile visible after the game is lost

Example:

```text
. . . 1
_ F 1 .
_ 2 3 _
_ _ _ _
```

Interpret the array using zero-based coordinates:

- `x` is the column index
- `y` is the row index
- `board[y][x]` is the tile value

Tile state meanings:

- `hidden`: unrevealed and unflagged
- `flagged`: currently flagged as a mine candidate
- `revealed`: safely revealed
- `mine`: appears only in terminal loss state

Interpretation of `adjacent_mines`:

- If `revealed`, the value is the number of adjacent mines
- If `0`, the tile has no adjacent mines
- If hidden or flagged during active play, `adjacent_mines` may be `null`

## Strategy Guidance

- Prefer moves that are logically certain
- Use revealed numbers to infer safe tiles and mine tiles
- Flag tiles only when there is strong justification, because incorrect flags lose points
- Prefer guaranteed safe reveals over speculative flags when uncertainty is high
- Use the safe first move to open information quickly
- Track local constraints around numbered tiles
- Avoid random reveals unless no deterministic move exists
- If forced to guess, choose the move with the lowest estimated mine risk
- Output only required json

## Decision Policy

For a given board state:
1. Read the full visible board state
2. interpret `board[y][x]` using the compact symbol rules
3. Identify deterministic safe reveals
4. Identify deterministic mine flags
5. If no deterministic move exists, estimate the least risky hidden tile
6. Return exactly one move
'''



#### Define Custom Chat Template

In [ ]:
chat_template = '''
{%- if messages[0].role == 'system' %}
    {{- '<|im_start|>system\n' + messages[0].content + '<|im_end|>\n' }}
{%- endif %}

{%- for message in messages %}
    {%- if message.content is string %}
        {%- set content = message.content %}
    {%- else %}
        {%- set content = '' %}
    {%- endif %}

    {%- if message.role == "user" %}
        {{- '<|im_start|>user\n' + content + '<|im_end|>\n' }}

    {%- elif message.role == "system" and not loop.first %}
        {{- '<|im_start|>system\n' + content + '<|im_end|>\n' }}

    {%- elif message.role == "assistant" %}
        {%- set reasoning_content = '' %}

        {%- if message.reasoning_content is string %}
            {%- set reasoning_content = message.reasoning_content %}
        {%- else %}
            {%- if '</think>' in content %}
                {%- set reasoning_content = content.split('</think>')[0].rstrip('\n').split('<think>')[-1].lstrip('\n') %}
                {%- set content = content.split('</think>')[-1].lstrip('\n') %}
            {%- endif %}
        {%- endif %}

        {%- if reasoning_content %}
            {{- '<|im_start|>assistant\n<think>\n' + reasoning_content.strip('\n') + '\n</think>\n\n' + content.lstrip('\n') + '<|im_end|>\n' }}
        {%- else %}
            {{- '<|im_start|>assistant\n' + content + '<|im_end|>\n' }}
        {%- endif %}
    {%- endif %}
{%- endfor %}

{%- if add_generation_prompt %}
    {{- '<|im_start|>assistant\n' }}
{%- endif %}
'''

tokenizer.chat_template = chat_template


#### Load The Datasets for SFT

In [ ]:
from pathlib import Path
import pandas as pd
train = pd.read_csv('~/dataset/dataset_train.csv')
test = pd.read_csv('~/dataset/dataset_test.csv')

#### Chat Template Example

In [ ]:
user = str(f"Board State: {train['input'][0]}\nMax Mines: {train['max_mines'][0]}\nMax Rows: {train['rows'][0]}\nMax Columns {train['columns'][0]}")
assistant = train['output'][0]

tokenizer.apply_chat_template(
    [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user},
        {"role": "assistant", "content": assistant},
    ],
    tokenize = False,
    add_generation_prompt = True,
)



### Format the dataset

In [ ]:
def format_dataset(df):
    input = df['input']
    output = df['output']

    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": input},
        {"role": "assistant", "content": output},
    ]

train['Messages'] = train.apply(format_dataset, axis=1)
test['Messages'] = test.apply(format_dataset, axis=1)

#check the formatted messages
train['Messages'][0]

#### Convert to HuggingFace compatible dataset

In [ ]:
from datasets import Dataset
# train_small_batch = train[0:300]
# test_small_batch = test[0:300]

# train_small_batch["text"] = tokenizer.apply_chat_template(train_small_batch["Messages"].values.tolist(), tokenize = False, add_generation_prompt = False)
# test_small_batch["text"] = tokenizer.apply_chat_template(test_small_batch["Messages"].values.tolist(), tokenize = False, add_generation_prompt = False)

# train_small_batch = Dataset.from_pandas(train_small_batch)
# test_small_batch = Dataset.from_pandas(test_small_batch)

In [ ]:
del train
import gc
torch.cuda.empty_cache()
gc.collect()

### Start the Finetuning-Process

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_small_batch,
    args = SFTConfig(
        dataset_text_field = "text",
        dataset_num_proc = 1,
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 2, # Use GA to mimic batch size!
        warmup_steps = 5,
        num_train_epochs = 2, # Set this for 1 full training run.
        learning_rate = 2e-4, # Reduce to 2e-5 for long training runs
        logging_steps = 2,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 42,
        report_to = "none", # Use TrackIO/WandB etc
    ),
)

In [ ]:
trainer.train()

In [ ]:
test_message = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": test['input'][7]}
]

text = tokenizer.apply_chat_template(
    test_message,
    tokenize = False,
    add_generation_prompt = True, # Must add for generation
)
text

In [ ]:
from transformers import TextStreamer
_ = model.generate(
    **tokenizer(text, return_tensors = "pt").to("cuda"),
    temperature = 0,
    max_new_tokens = 64,
    streamer = TextStreamer(tokenizer, skip_prompt = False),
)

#### Save as .safetensors 16-bit

In [ ]:
import os
HF_USERNAME = os.getenv('HF_USERNAME', '')
HF_TOKEN = os.getenv('HF_TOKEN')

if False:
    model.save_pretrained_merged("qwen_finetune_16bit", tokenizer, save_method = "merged_16bit",)
    model.push_to_hub_merged(f"{HF_USERNAME}/qwen_finetune_16bit", tokenizer, save_method = "merged_16bit", token = HF_TOKEN)



#### save as GGUF

In [ ]:
if False:
    model.save_pretrained_gguf("qwen_finetune", tokenizer,)
    model.push_to_hub_gguf(f"{HF_USERNAME}/qwen_finetune", tokenizer, token = HF_TOKEN, quantization_method = 'q8_0')


### Reinforcement Learning with GRPO

In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048
lora_rank = 32
model_path = "/teamspace/studios/this_studio/model_weights/Qwen3 4B SFT/"
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_path,
    max_seq_length = max_seq_length,
    load_in_4bit = True, # False for LoRA 16bit
    fast_inference = False, # Enable vllm fast inference
    max_lora_rank = lora_rank,
    gpu_memory_utilization = 0.75, # Reduce if out of memory
)


model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = lora_rank * 2, # *2 speeds up training
    use_gradient_checkpointing = "unsloth", # Reduces memory usage
    random_state = 42,
)




#### Define Reward Function


In [ ]:
import random
import requests
import sys
from pathlib import Path
from datasets import Dataset
from torch.utils.data import IterableDataset
import torch
import ast
import requests


base_dir = Path.cwd().parent
print("Notebook cwd:", base_dir)

project_dir = base_dir / "src"
print("Project dir:", project_dir)

if not project_dir.exists():
    raise FileNotFoundError(f"Folder not found: {project_dir}")

# Add the project folder to Python path
if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))

print("Added to sys.path:", sys.path[0])

from minesweeper.engine import GameConfig, GameEngine

# API_BASE_URL = "http://127.0.0.1:8000"

#Create the game first
def sample_prefill_move(game: GameEngine, rng: random.Random, reveal_probability=0.7):
    reveal_candidates = [tile for tile in game.iter_tiles() if not tile.is_revealed and not tile.is_flagged]
    flag_candidates = [tile for tile in game.iter_tiles() if not tile.is_revealed and not tile.is_flagged]

    if not reveal_candidates and not flag_candidates:
        return None

    if reveal_candidates and (not flag_candidates or rng.random() < reveal_probability):
        tile = rng.choice(reveal_candidates)
        return {"action": "reveal", "x": tile.x, "y": tile.y}

    tile = rng.choice(flag_candidates)
    return {"action": "flag", "x": tile.x, "y": tile.y}


def create_game(width=9, height=9, mine_density=0.15, seed=None, output_format="compact", max_prefill_moves=4):
    rng = random.Random(seed) if seed is not None else random.Random()
    game = GameEngine(config=GameConfig(width=width, height=height, mine_density=mine_density), rng=rng)

    prefill_moves = rng.randint(0, max_prefill_moves)

    if prefill_moves > 0:
        first_x = rng.randrange(width)
        first_y = rng.randrange(height)
        game.reveal(first_x, first_y)

        for _ in range(prefill_moves - 1):
            if game.status.value != "in_progress":
                break

            move = sample_prefill_move(game, rng)
            if move is None:
                break

            try:
                if move["action"] == "reveal":
                    game.reveal(move["x"], move["y"])
                else:
                    game.flag(move["x"], move["y"])
            except ValueError:
                continue

    visible_state = game.compact_state()
    full_board_state = game.full_board_compact_state() if game.snapshot()["mines_placed"] else None
    return [visible_state, full_board_state, game.snapshot()]

#create a batch of games with random configurations
def create_game_batch(num_games = 5):
    board_size = [(5, 5), (9, 9), (12, 12), (15, 15), (15, 20), (12, 15), (20, 20)]
    mine_density = [0.15, 0.30]
    
    games = []
    for _ in range(num_games):
        width, height = random.choice(board_size)
        density = random.choice(mine_density)
        game = create_game(width=width, height=height, mine_density=density)
        games.append(game)
    
    return games

#Helper function to build user prompt from game state
def build_user_prompt_from_state(game):
    user_prompt = f"Game State: {game['board']} \nTotal rows: {game['height']} \nTotal columns: {game['width']} \nTotal mines: {game['mine_count']} "
    return user_prompt


def create_dataset_from_games(games):
    rows = []
    # Set to None to use raw user prompts without chat template formatting
    for game in games:
        user_prompt = build_user_prompt_from_state(game[0])
        rows.append(
            {
                "prompt": tokenizer.apply_chat_template(
                    [
                        {"role": "system", "content": system_prompt},
                        {"role": "user", "content": user_prompt}
                    ],
                    tokenize=False,
                    add_generation_prompt=True
                ) ,
                "revealed_board": str(game[1]),
                "snapshot": str(game[2]),
            }
        )
    return Dataset.from_list(rows)

game = create_game()
print(game)  
# import pandas as pd
  
# games = create_game_batch(num_games=5)
# dataset = create_dataset_from_games(games)
# print("Dataset created with", len(dataset), "games.")

# df = dataset.to_pandas()
# print(df.head())

class MinesweeperDataset(IterableDataset):
    def __init__(self, board_sizes, mine_densities, max_prefill_moves=4, seed=None):
        self.board_sizes = board_sizes
        self.mine_densities = mine_densities
        self.max_prefill_moves = max_prefill_moves
        self.seed = seed
        
    
    def __iter__(self):
        rng = random.Random(self.seed)
        while True:
            width, height = rng.choice(self.board_sizes)
            density = rng.choice(self.mine_densities)
            game_seed = rng.randrange(2**31)
            game = create_game(width=width, height=height, mine_density=density, seed=game_seed, max_prefill_moves=self.max_prefill_moves)
            user_prompt = build_user_prompt_from_state(game[0])
            yield {
                "prompt": tokenizer.apply_chat_template(
                    [
                        {"role": "system", "content": system_prompt},
                        {"role": "user", "content": user_prompt}
                    ],
                    tokenize=False,
                    add_generation_prompt=True 
                ),
                "board_state": game[0]['board'],
                "game_id": game[0]['game_id'],
                "rows": game[0]['height'],
                "columns": game[0]['width'],
                "max_mines": game[0]['mine_count'],
                "revealed_board": str(game[1]),
                "snapshot": str(game[2]),
            }
            
    # str(f"Board State: {train['input'][0]}\nMax Mines: {train['max_mines'][0]}\nMax Rows: {train['rows'][0]}\nMax Columns {train['columns'][0]}")

board_size = [(5, 5), (9, 9), (12, 12), (15, 15), (15, 20), (12, 15), (20, 20)]
mine_density = [0.15, 0.30]
live_train_dataset = MinesweeperDataset(board_sizes=board_size, mine_densities=mine_density, max_prefill_moves=4, seed=42)

In [ ]:
import ast
import json
import re

rewards = []

def build_game_from_state(snapshot):
    game = GameEngine.from_snapshot(ast.literal_eval(snapshot))
    return game

def validate_move(rows, columns, response):
    action = response.get("action")
    x = response.get("x")
    y = response.get("y")

    if action not in {"reveal", "flag"}:
        return False
    if not isinstance(x, int) or not isinstance(y, int):
        return False
    if x < 0 or x >= columns or y < 0 or y >= rows:
        return False
    return True


def apply_move_and_get_reward(game: GameEngine, response: dict):
    before_score = game.score
    action = response["action"]
    x = response["x"]
    y = response["y"]
    target_tile = game.get_tile(x, y)

    if action == "flag" and target_tile.is_flagged:
        raise ValueError("Flagging an already flagged tile is penalized in GRPO reward shaping.")

    if action == "reveal":
        game.reveal(x, y)
    else:
        game.flag(x, y)

    return float(game.score - before_score)
    

def calculate_reward(prompts=None, completions=None, **kwargs):
    rows_list = kwargs.get("rows") or []
    columns_list = kwargs.get("columns") or []
    snapshots = kwargs.get("snapshot") or []

    rewards = []

    for i, completion in enumerate(completions or []):
        reward = 0.0

        rows = rows_list[i] if i < len(rows_list) else None
        columns = columns_list[i] if i < len(columns_list) else None
        snapshot = snapshots[i] if i < len(snapshots) else None

        if snapshot is None or rows is None or columns is None:
            raise ValueError(f"Missing required snapshot information for reward calculation at index {i}")

        try:
            response = ast.literal_eval(completion.strip())
        except Exception:
            rewards.append(-6.0)
            continue

        try:
            game = build_game_from_state(snapshot)
        except Exception:
            rewards.append(-10.0)
            continue

        if not validate_move(rows, columns, response):
            rewards.append(-6.0)
            continue

        try:
            reward = apply_move_and_get_reward(game, response)
        except Exception:
            reward -= 6.0

        rewards.append(reward)

    return rewards

#### Set GRPO config and sampling parameters

In [ ]:
from vllm import SamplingParams
from trl import GRPOConfig, GRPOTrainer

vllm_sampling_params = SamplingParams(
    min_p=0.1,
    top_p=1.0,
    top_k=-1,
    seed=42,
    stop=[tokenizer.eos_token],
    include_stop_str_in_output=True,
)

training_args = GRPOConfig(
    vllm_sampling_params=vllm_sampling_params,
    temperature=1.0,
    learning_rate=5e-6,
    weight_decay=0.001,
    warmup_ratio=0.1,
    lr_scheduler_type="linear",
    optim="adamw_8bit",
    logging_steps=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    num_generations=4,
    max_prompt_length=max_seq_length - 100,
    max_completion_length=100,
    max_steps=100,
    save_steps=100,
    report_to="none",
    output_dir="outputs",
)


In [ ]:
# GRPO trainer using live engine states instead of the offline CSV dataset
trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[calculate_reward],
    args=training_args,
    train_dataset=live_train_dataset,
    # eval_dataset=live_eval_dataset,
)

trainer.train()


### Play Live Game Via API

In [ ]:
def create_api_game(width=9, height=9, mine_density=0.15, seed=None, output_format="compact"):
    payload = {
        "width": width,
        "height": height,
        "mine_density": mine_density,
        "seed": seed,
        "output_format": output_format,
    }
    response = requests.post(f"{API_BASE_URL}/games", json=payload, timeout=30)
    response.raise_for_status()
    return response.json()


def fetch_api_state(game_id, output_format="compact"):
    response = requests.post(
        f"{API_BASE_URL}/games/{game_id}/state",
        json={"output_format": output_format},
        timeout=30,
    )
    response.raise_for_status()
    return response.json()


def submit_api_move(game_id, action, x, y, output_format="compact"):
    response = requests.post(
        f"{API_BASE_URL}/games/{game_id}/moves",
        json={"action": action, "x": x, "y": y, "output_format": output_format},
        timeout=30,
    )
    response.raise_for_status()
    return response.json()


def generate_move_from_local_model(state, model, tokenizer, system_prompt_text=system_prompt, max_new_tokens=96):
    messages = [
        {"role": "system", "content": system_prompt_text},
        {"role": "user", "content": build_user_prompt_from_state(state)},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    device = next(model.parameters()).device
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    completion_tokens = output[0][inputs["input_ids"].shape[1]:]
    completion_text = tokenizer.decode(completion_tokens, skip_special_tokens=True).strip()
    move = parse_move(completion_text)
    if move is None:
        raise ValueError(f"Model output could not be parsed into a move: {completion_text!r}")
    return move, completion_text


def play_game_via_api(
    model,
    tokenizer,
    width=9,
    height=9,
    mine_density=0.15,
    seed=None,
    max_turns=200,
    verbose=True,
):
    state = create_api_game(width=width, height=height, mine_density=mine_density, seed=seed, output_format="compact")
    game_id = state["game_id"]

    if verbose:
        print(f"Created game_id={game_id}")
        print(render_compact_board(state["board"]))

    history = []
    for turn in range(1, max_turns + 1):
        move, raw_text = generate_move_from_local_model(state, model=model, tokenizer=tokenizer)
        next_state = submit_api_move(game_id, move["action"], move["x"], move["y"], output_format="compact")
        history.append({
            "turn": turn,
            "prompt_board": state["board"],
            "raw_model_output": raw_text,
            "move": move,
            "result": next_state.get("last_move"),
            "status": next_state["status"],
            "score": next_state["score"],
        })

        if verbose:
            last_move = next_state.get("last_move", {})
            print(f"Turn {turn}: {move} -> {last_move.get('message')} score_delta={last_move.get('score_delta')}")
            print(render_compact_board(next_state["board"]))
            print(f"status={next_state['status']} score={next_state['score']}\n")

        state = next_state
        if state["status"] in {"won", "lost"}:
            break

    return {"game_id": game_id, "final_state": state, "history": history}


# Example usage after the API server is running on port 8000:
# result = play_game_via_api(model, tokenizer, width=9, height=9, mine_density=0.15, seed=7)
# result["final_state"]["status"], result["final_state"]["score"]
